<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2025/2026  

<img src="../../img/portada.png" width="300"/>

---

#  Aprendizaje automático relacional  
### Primera Convocatoria – Junio 2026  

**Profesor:** Pedro Almagro Blanco

**Autores:**  
David Espina Apellaniz  
Marco Padilla Gómez  

**Fecha de entrega:** 8 de Junio de 2025  

</div>

# 07 - Comparativa final de modelos

En este notebook se reúnen los resultados finales de los modelos entrenados y se selecciona el mejor modelo para la tarea de clasificación de usuarios de Twitch según `mature`.

La comparación se realiza usando principalmente la métrica `f1`, aunque también se conservan accuracy, precision y recall.

## Objetivo metodologico

Este cuaderno reune los resultados de los modelos ajustados y selecciona el modelo final. La seleccion se hace con `f1`, porque en esta tarea interesa equilibrar falsos positivos y falsos negativos sobre la clase `mature`.


## 1. Importación de librerías y configuración inicial

Se cargan las funciones necesarias para leer modelos, guardar el modelo final y generar el gráfico comparativo.

In [1]:
from pathlib import Path
import sys

import pandas as pd

sys.path.append("../../")

from src.evaluation import load_model, save_model
from src.visualization import save_model_comparison_plot

ROOT = Path("../../").resolve()
RESULTS_DIR = ROOT / "results"
MODELS_DIR = ROOT / "models"
IMG_DIR = ROOT / "img"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Carga y ordenación de resultados

Se cargan las métricas de los experimentos de hiperparámetros de Decision Tree, Random Forest y kNN, y se ordenan por `f1`.

### Criterio de seleccion

La tabla se ordena de mayor a menor `f1`. El primer modelo se guarda como `mejor_modelo_twitch.joblib`, cumpliendo el requisito de que el modelo seleccionado quede claramente indicado y pueda cargarse posteriormente.


In [2]:
results = pd.concat([
    pd.read_csv(RESULTS_DIR / "dt_grid_metrics.csv"),
    pd.read_csv(RESULTS_DIR / "rf_grid_metrics.csv"),
    pd.read_csv(RESULTS_DIR / "knn_grid_metrics.csv"),
], ignore_index=True)
results = results.sort_values("f1", ascending=False).reset_index(drop=True)
results

,model,max_depth,accuracy,precision,recall,f1,n_estimators,n_neighbors,weights,metric
0,decision_tree,3.0,0.569892,0.377863,0.727941,0.497487,NaN,NaN,NaN,NaN
1,decision_tree,5.0,0.621505,0.402439,0.606618,0.483871,NaN,NaN,NaN,NaN
2,random_forest,10.0,0.702151,0.490040,0.452206,0.470363,200.0,NaN,NaN,NaN
3,random_forest,10.0,0.701075,0.488000,0.448529,0.467433,100.0,NaN,NaN,NaN
4,decision_tree,10.0,0.602151,0.368280,0.503676,0.425466,NaN,NaN,NaN,NaN
5,decision_tree,NaN,0.624731,0.358974,0.360294,0.359633,NaN,NaN,NaN,NaN
6,knn,NaN,0.669892,0.413793,0.308824,0.353684,NaN,3.0,uniform,euclidean
7,knn,NaN,0.667742,0.408867,0.305147,0.349474,NaN,3.0,distance,euclidean
8,knn,NaN,0.684946,0.437126,0.268382,0.332574,NaN,5.0,distance,euclidean
9,knn,NaN,0.688172,0.444444,0.264706,0.331797,NaN,5.0,uniform,euclidean


## 3. Guardado de tabla y gráfico comparativo

Se guarda la tabla final de métricas y un gráfico comparativo de los modelos.

In [3]:
results.to_csv(RESULTS_DIR / "metricas_modelos.csv", index=False)
save_model_comparison_plot(results, IMG_DIR / "comparativa_modelos.png")

<Figure size 1000x600 with 0 Axes>

## 4. Selección y guardado del mejor modelo

Se identifica el mejor modelo según `f1` y se guarda como modelo final del trabajo.

In [4]:
best_row = results.iloc[0]
if best_row["model"] == "decision_tree":
    selected_model = load_model(MODELS_DIR / "decision_tree_grid.joblib")
elif best_row["model"] == "random_forest":
    selected_model = load_model(MODELS_DIR / "random_forest_grid.joblib")
else:
    selected_model = load_model(MODELS_DIR / "knn_grid.joblib")
save_model(selected_model, MODELS_DIR / "mejor_modelo_twitch.joblib")
best_row

model           decision_tree
max_depth                 3.0
accuracy             0.569892
precision            0.377863
recall               0.727941
f1                   0.497487
n_estimators              NaN
n_neighbors               NaN
weights                   NaN
metric                    NaN
Name: 0, dtype: object

## 5. Conclusiones

Este notebook resume la fase experimental del proyecto. El modelo seleccionado será el que se describa como propuesta final en la memoria y en la presentación.